# Advisor: 작업 중인 agent가 turn 도중 더 강력한 model에 자문하게 하기

모든 turn을 가장 뛰어난 model로 실행하는 것은 안전하지만 비용이 높은 선택입니다. 중간 등급 model을 사용하면 저렴하지만, 작업에서 더 깊은 판단이 필요한 단 하나의 결정에 agent가 부딪힐 수 있고 그때는 이미 결정을 내린 뒤입니다. 일반적인 해결책은 직접 escalation을 구현하는 것입니다. 어려운 사례를 감지하고, context를 묶고, 더 큰 model을 호출한 뒤 답을 다시 끼워 넣습니다.

advisor는 이를 세션 내부에서 수행합니다. agent의 `multiagent` roster에 `{"type": "advisor", "model": ...}` 항목으로 두 번째, 더 뛰어난 model을 지정합니다. 그러면 primary thread에 turn 도중 호출할 수 있는 `advisor` tool이 생깁니다. 플랫폼은 지금까지의 대화를 advisor model에 제공하고, 조언을 작업 model에 반환하며, sampling이 계속됩니다. 각 자문은 자체 token 사용량을 가진 짧게 유지되는 플랫폼 thread로 실행되므로 언제 이루어졌는지 확인하고 정확한 비용을 계산할 수 있습니다.

이 notebook에서는 다음을 살펴봅니다.

- agent에 advisor 항목 추가하기
- 작업 model에 자문할 시점 알려 주기
- event stream에서 자문을 확인하고 thread에서 비용 읽기
- advisor의 출력이 redacted 상태로 돌아올 때 달라지는 점


## 1. client 설정하기

advisor 항목은 표준 `managed-agents-2026-04-01` beta header를 사용하는 `multiagent` block에 있습니다. 타입이 지정된 roster 형태를 사용하려면 `anthropic>=0.121.0`이 필요합니다.


In [1]:
%%capture
%pip install -qU "anthropic>=0.121.0" python-dotenv

In [2]:
import os

import anthropic
from dotenv import load_dotenv

load_dotenv()

BETAS = ["managed-agents-2026-04-01"]
WORKER_MODEL = os.environ.get("COOKBOOK_MODEL", "claude-sonnet-5")
ADVISOR_MODEL = os.environ.get("COOKBOOK_ADVISOR_MODEL", "claude-opus-5")
client = anthropic.Anthropic()

## 2. Sonnet agent에 Opus advisor 제공하기

advisor는 tool 정의나 별도로 생성하는 agent가 아니라 roster에 선언합니다. 항목은 specialist와 함께 `multiagent.agents`에 들어가며, advisor 항목만 있는 roster도 유효합니다. agent는 자문하지만 subagent를 생성하지는 않습니다. 서버가 저장 시 적용하는 규칙은 다음과 같습니다.

- roster당 advisor 항목은 최대 하나입니다.
- `model`은 advisor로 허용되어야 하며, agent 자체 model과의 조합도 플랫폼이 허용하는 것이어야 합니다. 부적격 조합은 runtime에서 뜻밖의 문제가 되는 대신 `agents.create`에서 400을 반환합니다.
- 이 항목은 roster에서 `anthropic.advisor`라는 이름을 차지하므로 specialist가 이 이름을 사용할 수 없습니다.

runtime에서 플랫폼은 작업 model에 이 항목을 `advisor` tool로 제공합니다. advisor는 작업 model이 작성한 query가 아니라 호출 시점까지의 전체 대화를 읽으므로 tool은 입력을 받지 않습니다. 따라서 자문 정책을 설정하는 곳은 system prompt입니다. 이 정책은 model이 언제 tool을 사용하는지를 제어합니다.


In [3]:
SYSTEM = """You are a backend engineer designing HTTP APIs.
You have an advisor: a more capable model that can review your conversation so far and
send back guidance. Consult it with the advisor tool before you commit to any decision
that would be expensive to reverse once clients depend on it: identifier and idempotency
schemes, pagination contracts, error semantics, versioning. Do routine drafting yourself.
When you consult, act on the guidance you get back and say what you changed.
Write your final design to /mnt/session/outputs/design.md."""

designer = client.beta.agents.create(
    name="api_designer",
    description="Designs HTTP APIs, escalating irreversible decisions to an advisor.",
    # 이 agent가 어느 cookbook에서 왔는지 Anthropic에 알립니다. 제거해도 안전합니다.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-consult-an-advisor"},
    model={"id": WORKER_MODEL},
    system=SYSTEM,
    tools=[{"type": "agent_toolset_20260401"}],
    multiagent={
        "type": "coordinator",
        "agents": [
            {"type": "advisor", "model": ADVISOR_MODEL},
        ],
    },
    betas=BETAS,
)
print(f"{designer.name}: {designer.id} v{designer.version}")
print("roster:", [entry.to_dict() for entry in designer.multiagent.agents])

api_designer: agent_staging_018FvRjvrM6ASGuXXcDTAvdH v1
roster: [{'model': 'claude-opus-5', 'type': 'advisor'}]


## 3. 어려운 판단이 포함된 작업 맡기기

자금을 이동하는 endpoint의 idempotency 설계는 prompt가 agent에 escalation하도록 지시한 종류의 결정입니다. 작성은 저렴하지만 client가 이를 기반으로 출시된 뒤 변경하기에는 비용이 큽니다. 나머지 설계는 일상적이므로 판단이 잘 조정된 agent는 이 부분에서 자문하고 나머지는 직접 작성합니다.

자문 지출은 자문하는 turn에 의해서만 제한됩니다. advisor token은 작업 model의 token에 더하여 청구되며 자문별 상한은 없습니다. 세션에는 `budget`이 설정되어 있어 지나치게 잦은 escalation 습관이 청구액을 계속 늘리는 대신 알려진 상한에서 일시 정지합니다(작동 방식은 [`CMA_cap_session_spend.ipynb`](CMA_cap_session_spend.ipynb)에 있습니다).


In [4]:
env = client.beta.environments.create(
    name="advisor-demo",
    config={"type": "anthropic_cloud", "networking": {"type": "unrestricted"}},
    betas=BETAS,
)

session = client.beta.sessions.create(
    agent=designer.id,
    environment_id=env.id,
    title="Design: refunds API",
    budget={"type": "limit", "max_list_cost": {"currency": "USD", "amount": "500"}},
    initial_events=[
        {
            "type": "user.message",
            "content": [
                {
                    "type": "text",
                    "text": "Design the REST surface for a refunds resource on a payments "
                    "API: create, retrieve, list. Refund creation is retried by mobile "
                    "clients on flaky networks, and a double refund costs real money. "
                    "Cover the request/response shapes, idempotency, pagination on list, "
                    "and error codes.",
                }
            ],
        }
    ],
    betas=BETAS,
)
print(session.id, session.status)

sesn_staging_013TqQwzrbUZDX1KJW2b9KnC running


## 4. primary stream에서 자문 관찰하기

자문은 `agent.tool_use` event가 아닙니다. primary stream에 `agent_name`이 `anthropic.advisor`로 설정된 thread lifecycle로 나타납니다.

```
session.thread_created            agent_name: anthropic.advisor
session.thread_status_running     agent_name: anthropic.advisor
agent.thread_message_received     from_agent_name: anthropic.advisor, content: [{type: "text", ...}]
session.thread_status_idle        agent_name: anthropic.advisor
session.thread_status_terminated  agent_name: anthropic.advisor
```

자문을 요청한 thread는 내내 `running` 상태를 유지합니다. 조언은 호출 시점에 전달되고 sampling이 계속되므로 외부에서는 primary thread가 잠시 멈추고 advisor thread가 나타났다가 사라진 뒤, primary thread가 조언을 반영하여 재개하는 것으로 보입니다. 각 자문은 전달을 완료하면 스스로 종료하는 새 thread입니다.


In [5]:
def advice_text(content):
    """advisor 전달을 표시합니다. text를 합치거나 redacted인 경우 표시 문구를 사용합니다."""
    if any(block.type == "redacted" for block in content):
        return "[redacted advice: content withheld by the advisor model's policy]"
    return "".join(block.text for block in content if block.type == "text")


consultations = 0

with client.beta.sessions.events.stream(session.id, betas=BETAS) as stream:
    for ev in stream:
        if ev.type == "session.thread_created" and ev.agent_name == "anthropic.advisor":
            consultations += 1
            print(f"\n--- consulting advisor #{consultations} ({ev.session_thread_id}) ---")
        elif (
            ev.type == "agent.thread_message_received" and ev.from_agent_name == "anthropic.advisor"
        ):
            print(advice_text(ev.content)[:800])
            print("--- advice received, primary thread resumes ---\n")
        elif ev.type == "agent.message":
            print("".join(b.text for b in ev.content if b.type == "text"), end="")
        elif ev.type == "session.status_idle":
            print(f"\n[idle] stop_reason={ev.stop_reason.type}")
            break

print(f"\nconsultations this turn: {consultations}")


--- consulting advisor #1 (sthr_staging_01PnQEQPc1nLxUK4S2vcVbCL) ---


[redacted advice: content withheld by the advisor model's policy]
--- advice received, primary thread resumes ---



Incorporated the advisor's feedback directly into the design — the most consequential changes were:

- **Idempotency became a stored resource with a claim→execute→finalize lifecycle**, not a simple header check. This closes the "processor succeeded but our server died before recording it" window, which a naive check-then-execute design leaves open.
- **Added a second, independent safety layer**: a server-side invariant (`sum(active refunds) + new_amount <= captured_amount`) enforced under a row lock, so the system is safe even if a client reuses/loses its idempotency key.
- **Made refund creation explicitly asynchronous** (`201` + `status: pending`, terminal states `succeeded|failed|canceled`) instead of implying synchronous success.
- **Added a recovery path** (`GET /refunds?idempotency_key=...`) for clients that lose the response to a request that actually succeeded — critical for flaky mobile networks.
- **Switched list pagination to opaque cursor-based** instead of offset, since re

The design is written to `/mnt/session/outputs/design.md`.

**Summary of the approach:**

- **Endpoints:** `POST /v1/payments/{payment_id}/refunds` (create, scoped to parent), `GET /v1/refunds/{id}` (retrieve), `GET /v1/refunds` (list) — top-level for retrieve/list per advisor guidance.
- **Idempotency — the centerpiece:** `Idempotency-Key` header is required and persisted as its own resource with a **claim → execute → finalize** lifecycle (claim written in the same transaction as the pending refund), so a crash after the processor succeeds but before we record it can't produce a silent duplicate on retry. Replay rules are explicit: same key+body replays the original response, same key+different body is a `422`, same key still in-flight is a `409`.
- **Second safety net, independent of the header:** a server-enforced invariant (`sum(active refunds) + new_amount <= captured_amount`) under a row lock, protecting against buggy clients that regenerate keys or omit them.
- **Async-first sta

## 5. 자문 비용 계산하기

각 자문은 실제 세션 thread이므로 비용이 primary thread 수치에 보이지 않게 합쳐지는 대신 해당 thread에 나타납니다. 각 advisor thread를 `session.thread_created` event의 id로 가져와 `usage`를 읽습니다. token 수와 `usage.list_cost`가 있습니다. 후자는 advisor model의 공개 정가로 계산한 thread의 token 비용이며, USD의 최소 단위로 표현한 정수 문자열입니다(`"131"`은 $1.31). 따라서 직접 관리하는 요금표 대신 API에서 비용을 얻습니다. 이 token은 primary thread의 token에 더하여 청구되며, 자체 `list_cost`가 있는 세션 수준 `usage` 합계에 이미 포함되어 있습니다.

완료된 자문은 `terminated` thread입니다. 계속 조회할 수 있지만 `agent`는 일반적인 snapshot agent가 아닙니다. 플랫폼 advisor를 뒷받침하는 Agent resource가 없으므로 thread에는 roster 항목 자체인 `{"type": "advisor", "model": ...}`가 들어갑니다. 이름보다 `agent.type`의 `"advisor"` 값이 플랫폼 advisor thread와 일반적인 child를 구분하는 확실한 방법입니다.


In [6]:
from decimal import Decimal


def usd(money) -> str:
    """정수 최소 단위 금액("131" = $1.31)을 달러로 표시합니다."""
    return f"${Decimal(money.amount) / 100:.2f}"


advisor_thread_ids = [
    ev.session_thread_id
    for ev in client.beta.sessions.events.list(session.id, limit=1000, betas=BETAS)
    if ev.type == "session.thread_created" and ev.agent_name == "anthropic.advisor"
]

advisor_in = advisor_out = 0
advisor_cents = Decimal("0")
for thread_id in advisor_thread_ids:
    thread = client.beta.sessions.threads.retrieve(thread_id, session_id=session.id, betas=BETAS)
    usage = thread.usage
    advisor_in += usage.input_tokens or 0
    advisor_out += usage.output_tokens or 0
    advisor_cents += Decimal(usage.list_cost.amount)
    print(
        f"{thread.id}  agent.type={thread.agent.type}  model={thread.agent.model}  "
        f"status={thread.status}  "
        f"in={usage.input_tokens} out={usage.output_tokens}  {usd(usage.list_cost)}"
    )

total = client.beta.sessions.retrieve(session.id, betas=BETAS).usage
print(f"\nadvisor tokens: in={advisor_in} out={advisor_out}  cost=${advisor_cents / 100:.2f}")
print(
    f"session total:  in={total.input_tokens} out={total.output_tokens}  "
    f"cost={usd(total.list_cost)}  (advisor included)"
)

sthr_staging_01PnQEQPc1nLxUK4S2vcVbCL  agent.type=advisor  model=claude-opus-5  status=terminated  in=603 out=2468  $0.07

advisor tokens: in=603 out=2468  cost=$0.07
session total:  in=15553 out=8491  cost=$0.21  (advisor included)


## 6. 설계 읽기

agent는 `write` tool로 최종 문서를 작성했습니다. event log에서 가져와 자문이 반영된 idempotency 절을 살펴보세요.


In [7]:
design = ""
for ev in client.beta.sessions.events.list(session.id, limit=1000, betas=BETAS):
    if (
        ev.type == "agent.tool_use"
        and ev.name == "write"
        and ev.input.get("file_path", "").endswith("design.md")
    ):
        design = ev.input["content"]  # 수정이 있을 수 있으므로 마지막 write를 보관합니다

for line in design.splitlines():
    if line.startswith("#"):
        print(line)
print()
print(design[:1200])

# Refunds API Design
## 1. Resources & endpoints
## 2. Idempotency (the core problem: safe retries on flaky mobile networks)
### 2.1 Header contract
### 2.2 Server-side lifecycle (claim → execute → finalize)
### 2.3 Replay semantics (what a retry actually gets back)
### 2.4 Second, independent safety layer: a business invariant
### 2.5 Recovery path for lost responses
## 3. Request / response shapes
### 3.1 Create
### 3.2 Retrieve
### 3.3 List
## 4. Error model

# Refunds API Design

## 1. Resources & endpoints

| Action | Method & path |
|---|---|
| Create a refund | `POST /v1/payments/{payment_id}/refunds` |
| Retrieve a refund | `GET /v1/refunds/{refund_id}` |
| List refunds | `GET /v1/refunds` |
| Look up by idempotency key (recovery) | `GET /v1/refunds?idempotency_key={key}` |

**Why creation is nested but retrieve/list are top-level:** creation under the parent
gives a natural scope for the amount-conservation invariant ("refunds of this payment")
and a clean `404` when `payment_

## 7. 조언이 redacted 상태로 돌아올 때

자문 내용을 읽을 수 있는지는 advisor model의 속성으로, Messages API advisor tool에 적용되는 것과 같은 Anthropic 정책이 결정합니다. 정책에 따라 출력을 공개하지 않는 model에서는 전달 event에 text 대신 `[{"type": "redacted"}]` placeholder block이 들어가며, advisor thread 자체의 `agent.message` event도 placeholder입니다.

Redaction은 관찰할 수 있는 내용을 바꾸지만 agent가 사용하는 내용은 바꾸지 않습니다. 작업 model은 어느 경우든 조언 전체를 읽으므로 설계는 여전히 개선됩니다. 구조에 관한 모든 정보는 계속 보입니다. thread, 상태 전환, 시간 정보, `usage`가 그것이며, 이 때문에 4단계의 `advice_text` helper는 오류를 내는 대신 표시용 문구를 출력합니다. 어느 정책에서도 advisor 자체의 reasoning은 전달되지 않고 최종 text만 전달됩니다.

정책은 API 변경 없이 바뀔 수 있으므로 이 event를 소비하는 모든 곳에서 `text`와 `redacted`를 똑같이 정상적인 경우로 취급하세요.


## 8. specialist 팀과 함께 사용하는 advisor

advisor 항목은 일반 roster 구성원과 `agents` 배열을 공유하므로 coordinator는 같은 세션에서 specialist에 위임하고 더 강력한 model에 자문할 수 있습니다. 자문하는 것은 primary thread뿐입니다. 생성한 specialist는 자체 advisor 없이 실행되며, advisor thread는 child thread의 동시 실행 제한에서 제외되므로 이미 child 한도에 도달한 coordinator도 자문할 수 있습니다.

```python
client.beta.agents.create(
    name="research_coordinator",
    model={"id": WORKER_MODEL},
    system="Delegate research to your specialists, then synthesize their reports.",
    tools=[{"type": "agent_toolset_20260401"}],
    multiagent={
        "type": "coordinator",
        "agents": [
            "agent_01Res...",                                # id로 지정한 specialist
            {"type": "agent", "id": "agent_01Aud...", "version": 3},
            {"type": "advisor", "model": ADVISOR_MODEL},     # advisor
        ],
    },
    betas=BETAS,
)
```

coordinator 패턴 자체, 역할별 tool 범위 지정, 위임 event 타입은 [`CMA_coordinate_specialist_team.ipynb`](CMA_coordinate_specialist_team.ipynb)에서 다룹니다. advisor는 세션 생성 시 고정됩니다. agent에서 변경하면 이후 생성하는 세션에 영향을 주며, 이미 실행 중인 세션에는 영향을 주지 않습니다.


## 9. 정리하기


In [8]:
from utilities import wait_for_idle_status

wait_for_idle_status(client, session.id)
client.beta.sessions.archive(session.id, betas=BETAS)
client.beta.environments.archive(env.id, betas=BETAS)
print("archived")

archived
